# Design Desk Smoke Test

**What this is.** A functional smoke test of the Python stack pinned in `requirements.txt`. Each
pinned analysis package solves a problem whose answer is known independently, from a closed form,
a series solution or another package. The notebook compares the two with `math.isclose`. The last
cell counts the results and prints a statement of what ran and what matched. It is computed from
this run, not typed. If any check failed, the cell raises an error, so

```bash
MPLBACKEND=Agg jupyter nbconvert --to notebook --execute Design_Desk_Smoke_Test.ipynb --inplace
```

exits non-zero. Run it after every change to `requirements.txt`, and commit it with its outputs.

**What this is not.** It is not a design analysis. No problem is framed with the checklist, no
Assumption Ledger is kept, and no margin of safety is stated. The problems were chosen because
their answers are known exactly, not because they are realistic designs.

**Reading a result line.** `OK` or `FAIL`, then the package's value, the reference value, the
relative error and the tolerance. `SKIP` means the check needs a program pip can't install (GNU
Octave, OpenModelica) and none was found. A skipped check is reported as not covered and never
counts as a pass. A package that raises an error is recorded as `FAIL` with the error, and the
notebook carries on to the tally.

**Tolerances** come from what each method can achieve, not from what makes the check pass. They are
about 10⁻⁹ where the method is exact (cubic beam elements under a tip load, closed-form
identities), the mesh-study error where a field is discretized, and a small multiple of the solver
tolerance where an ODE is integrated.

**Units and signs.** SI (N, m, Pa, s), except Problem 6's shaft, which is stated in inch-pound units
as in Shigley. The packages disagree on sign conventions, so deflections and reactions are compared
as magnitudes. Each cell notes its package's convention.

| # | Problem | Packages |
|---|---|---|
| 0 | Environment: installed versions against the pins, imports, `pip check` | all pins |
| 1 | 2 m steel cantilever with a 1 kN tip load | SymPy, PlaneSections, Pynite, anaStruct, SymBeam, OpenSeesPy, CALFEM, EasyFEA, scikit-fem |
| 2 | 50 × 50 mm square section: second moment of area, torsion constant | sectionproperties |
| 3 | Thick-walled cylinder under internal pressure (Lamé) | scikit-fem |
| 4 | Mass-spring-damper: free vibration and frequency-domain properties | SciPy, python-control, PathSim |
| 5 | PID position loop around the same plant | PathSim, python-control |
| 6 | Desk calculations: shaft sizing, principal stresses, Castigliano, S–N fit, Weibull life | SciPy, pint, uncertainties, handcalcs |
| 7 | Rainflow counting and damage-equivalent load | wetb |
| 8 | AGMA spur-gear geometry | mechanism |
| 9 | WR-90 waveguide S-parameters | RapidFEM |
| 10 | The cantilever's stiffness matrix solved in the MATLAB language | oct2py and GNU Octave |
| 11 | OpenModelica model → FMU → FMPy, and the FMU in a PathSim loop | OMPython, FMPy, pathsim-fmi |

In [1]:
import math, sys, os, platform, datetime, shutil, subprocess, tempfile, warnings
import importlib, importlib.metadata as metadata
from contextlib import contextmanager
import numpy as np
import pandas as pd

RESULTS = []   # one row per check, tallied at the end


def _record(problem, name, status, got=np.nan, want=np.nan, unit="", error=np.nan, tol=np.nan, note=""):
    RESULTS.append(dict(problem=problem, check=name, status=status, got=got, want=want,
                        unit=unit, error=error, tol=tol, note=note))


def check(problem, name, got, want, rel_tol=1e-9, abs_tol=0.0, unit=""):
    """Compare a package's value with an independent reference value."""
    got, want = float(got), float(want)
    ok = math.isclose(got, want, rel_tol=rel_tol, abs_tol=abs_tol)
    if want != 0:
        error, tol, kind = abs(got - want) / abs(want), rel_tol, "rel"
    else:
        error, tol, kind = abs(got), abs_tol, "abs"
    status = "OK" if ok else "FAIL"
    _record(problem, name, status, got, want, unit, error, tol)
    print(f"{status:4s}  {name}: {got:.6g} vs {want:.6g} {unit}  ({kind} error {error:.1e}, tol {tol:.0e})")


def check_same(problem, name, got, want):
    """Exact comparison, for versions and flags."""
    status = "OK" if got == want else "FAIL"
    _record(problem, name, status, note=f"{got} vs {want}")
    print(f"{status:4s}  {name}: {got} vs {want}")


def skip(problem, name, reason):
    """A check that could not run. Reported as not covered, never as a pass."""
    _record(problem, name, "SKIP", note=reason)
    print(f"SKIP  {name}: {reason}")


@contextmanager
def guarded(problem, name):
    """Record an exception as a FAIL and carry on, so the tally still runs."""
    try:
        yield
    except Exception as e:
        _record(problem, name, "FAIL", note=f"raised {type(e).__name__}: {e}")
        print(f"FAIL  {name}: raised {type(e).__name__}: {e}")


print(f"Python {platform.python_version()} on {platform.system()} {platform.machine()}")

Python 3.11.17 on Linux x86_64


## 0. Environment

Three checks on the environment itself:

- **Every installed version matches its pin.** This catches a stale venv.
- **Every pinned package imports.** This is not the same thing. `pip check` reads only declared
  metadata, so a package can install cleanly and still fail on import, as SimuPy does on NumPy 2.
- **`pip check` reports no conflicts.**

`IMPORTS` maps each pin to its import name. A pin missing from this table fails, which keeps the
table in step with `requirements.txt`.

In [2]:
IMPORTS = {
    "numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sympy": "sympy", "pint": "pint", "forallpeople": "forallpeople",
    "uncertainties": "uncertainties", "handcalcs": "handcalcs",
    "planesections": "planesections", "PyNiteFEA": "Pynite", "anastruct": "anastruct",
    "symbeam": "symbeam", "sectionproperties": "sectionproperties",
    "openseespy": "openseespy.opensees", "calfem-python": "calfem.core",
    "EasyFEA": "EasyFEA", "rapidfem": "rapidfem", "scikit-fem": "skfem",
    "mechanism": "mechanism", "wetb": "wetb.fatigue_tools.fatigue",
    "control": "control", "pathsim": "pathsim", "oct2py": "oct2py",
    "fmpy": "fmpy", "pathsim-fmi": "pathsim_fmi", "OMPython": "OMPython",
    "jupyterlab": "jupyterlab", "nbconvert": "nbconvert", "ipykernel": "ipykernel",
    "playwright": "playwright",
}

pins = {}
with open("requirements.txt") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "==" in line:
            name, version = line.split("==")
            pins[name.split("[")[0].strip()] = version.strip()

for name, pinned in pins.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = "not installed"
    check_same(0, f"{name} version", installed, pinned)

for name in pins:
    module = IMPORTS.get(name)
    if module is None:
        _record(0, f"{name} import", "FAIL", note="no import name in IMPORTS")
        print(f"FAIL  {name} import: add its import name to IMPORTS")
        continue
    with guarded(0, f"{name} import"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        importlib.import_module(module)
        check_same(0, f"{name} import", "imported", "imported")

pip = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if "No module named pip" in pip.stderr:
    skip(0, "pip check", "pip is not installed in this environment")
else:
    print(pip.stdout.strip())
    check_same(0, "pip check", pip.returncode, 0)

OK    numpy version: 2.4.6 vs 2.4.6
OK    scipy version: 1.17.1 vs 1.17.1
OK    pandas version: 3.0.6 vs 3.0.6
OK    matplotlib version: 3.11.2 vs 3.11.2
OK    sympy version: 1.14.0 vs 1.14.0
OK    pint version: 0.25.3 vs 0.25.3
OK    forallpeople version: 3.0.0 vs 3.0.0
OK    uncertainties version: 3.2.3 vs 3.2.3
OK    handcalcs version: 1.11.0 vs 1.11.0
OK    planesections version: 1.4.2 vs 1.4.2
OK    PyNiteFEA version: 1.0.1 vs 1.0.1
OK    anastruct version: 1.7.0 vs 1.7.0
OK    symbeam version: 2.1.2 vs 2.1.2
OK    sectionproperties version: 3.10.2 vs 3.10.2
OK    openseespy version: 3.7.1.2 vs 3.7.1.2
OK    calfem-python version: 3.6.16 vs 3.6.16
OK    EasyFEA version: 7.0.0 vs 7.0.0
OK    rapidfem version: 0.24.0 vs 0.24.0
OK    scikit-fem version: 12.0.2 vs 12.0.2
OK    mechanism version: 1.1.10 vs 1.1.10
OK    wetb version: 0.1.33 vs 0.1.33
OK    control version: 0.10.2 vs 0.10.2
OK    pathsim version: 0.27.1 vs 0.27.1
OK    oct2py version: 6.1.1 vs 6.1.1
OK    fmpy version: 0

OK    sympy import: imported vs imported
OK    pint import: imported vs imported
OK    forallpeople import: imported vs imported
OK    uncertainties import: imported vs imported
OK    handcalcs import: imported vs imported


OK    planesections import: imported vs imported
OK    PyNiteFEA import: imported vs imported


OK    anastruct import: imported vs imported
OK    symbeam import: imported vs imported
OK    sectionproperties import: imported vs imported
OK    openseespy import: imported vs imported
OK    calfem-python import: imported vs imported


OK    EasyFEA import: imported vs imported
OK    rapidfem import: imported vs imported
OK    scikit-fem import: imported vs imported
OK    mechanism import: imported vs imported


...

octave: X11 DISPLAY environment variable not set
octave: disabling GUI features


OK    oct2py import: imported vs imported
OK    fmpy import: imported vs imported


OK    pathsim-fmi import: imported vs imported
OK    OMPython import: imported vs imported


OK    jupyterlab import: imported vs imported
OK    nbconvert import: imported vs imported
OK    ipykernel import: imported vs imported
OK    playwright import: imported vs imported


No broken requirements found.
OK    pip check: 0 vs 0


## 1. The shared benchmark: a cantilever with a tip load

A 2 m steel cantilever with a solid 50 × 50 mm section is built in at x = 0 and carries
P = 1 kN downward at the free end. This is the beam the README's CALFEM, EasyFEA and scikit-fem
sections use. Euler–Bernoulli theory gives

| quantity | closed form | value |
|---|---|---|
| tip deflection | PL³ / 3EI | 25.600 mm |
| tip rotation | PL² / 2EI | 19.2 mrad |
| wall reaction | P | 1000 N |
| wall moment | PL | 2000 N·m |

Every tool below models the beam with elements that are exact for a cubic deflection, so each
should match to round-off, not just to a mesh tolerance. Nine packages solving the same beam is
also a cross-check between them.

In [3]:
E = 200e9            # Young's modulus, steel [Pa]
nu = 0.3             # Poisson's ratio [-]
b = h = 0.050        # square section, 50 x 50 mm [m]
L = 2.0              # span [m]
P = 1000.0           # tip load, downward [N]
A = b * h            # area [m^2]
I = b * h**3 / 12    # second moment of area [m^4]

delta = P * L**3 / (3 * E * I)     # tip deflection [m]
theta = P * L**2 / (2 * E * I)     # tip rotation [rad]
print(f"I = {I:.6e} m^4   tip deflection {delta*1e3:.3f} mm   tip rotation {theta*1e3:.2f} mrad")

TIP = {}   # package -> computed tip deflection [mm], for the table at the end of this problem

I = 5.208333e-07 m^4   tip deflection 25.600 mm   tip rotation 19.20 mrad


In [4]:
# SymPy's singularity-function beam. Upward loads are positive, so the tip load is -P.
with guarded(1, "SymPy beam"):
    import sympy as sp
    from sympy.physics.continuum_mechanics.beam import Beam
    R0, M0 = sp.symbols("R0 M0")
    beam = Beam(L, E, I)
    beam.apply_load(R0, 0, -1)      # wall reaction
    beam.apply_load(M0, 0, -2)      # wall moment
    beam.apply_load(-P, L, -1)      # tip load
    beam.bc_deflection = [(0, 0)]
    beam.bc_slope = [(0, 0)]
    beam.solve_for_reaction_loads(R0, M0)
    v_tip = float(beam.deflection().subs(beam.variable, L))
    TIP["SymPy"] = abs(v_tip) * 1e3
    check(1, "SymPy tip deflection", abs(v_tip), delta, unit="m")
    check(1, "SymPy wall reaction", float(beam.reaction_loads[R0]), P, unit="N")
    check(1, "SymPy wall moment", abs(float(beam.reaction_loads[M0])), P * L, unit="N*m")

OK    SymPy tip deflection: 0.0256 vs 0.0256 m  (rel error 0.0e+00, tol 1e-09)
OK    SymPy wall reaction: 1000 vs 1000 N  (rel error 0.0e+00, tol 1e-09)
OK    SymPy wall moment: 2000 vs 2000 N*m  (rel error 0.0e+00, tol 1e-09)


In [5]:
# PlaneSections 1.4.2 (PyNiteFEA solver). Reactions are [Fx, Fy, M] on the restrained node.
with guarded(1, "PlaneSections"):
    import planesections as ps
    psbeam = ps.EulerBeam2D(np.linspace(0, L, 11), section=ps.SectionRectangle(E=E, d=h, w=b))
    psbeam.setFixity(0.0, ps.FixityTypes2D.getFixed())
    psbeam.addVerticalLoad(L, -P)
    ps.PyNiteAnalyzer2D(psbeam).runAnalysis()
    uy, _ = ps.getVertDisp(psbeam)          # returns (disp, x): displacement first
    TIP["PlaneSections"] = abs(uy[-1]) * 1e3
    check(1, "PlaneSections tip deflection", abs(uy[-1]), delta, unit="m")
    check(1, "PlaneSections wall reaction", psbeam.nodes[0].rFrc[1], P, unit="N")
    check(1, "PlaneSections wall moment", abs(psbeam.nodes[0].rFrc[2]), P * L, unit="N*m")

OK    PlaneSections tip deflection: 0.0256 vs 0.0256 m  (rel error 4.3e-13, tol 1e-09)
OK    PlaneSections wall reaction: 1000 vs 1000 N  (rel error 1.1e-12, tol 1e-09)
OK    PlaneSections wall moment: 2000 vs 2000 N*m  (rel error 6.1e-13, tol 1e-09)


In [6]:
# Pynite: a 3D frame model, so it needs G and a torsion constant J too (J doesn't affect this load).
with guarded(1, "Pynite"):
    from Pynite import FEModel3D
    frame = FEModel3D()
    frame.add_node("N1", 0, 0, 0)
    frame.add_node("N2", L, 0, 0)
    frame.add_material("Steel", E, E / (2 * (1 + nu)), nu, 7850)
    frame.add_section("Sq50", A, I, I, 0.1406 * b**4)
    frame.add_member("M1", "N1", "N2", "Steel", "Sq50")
    frame.def_support("N1", True, True, True, True, True, True)
    frame.add_node_load("N2", "FY", -P)
    frame.analyze()
    dy = frame.nodes["N2"].DY["Combo 1"]
    TIP["Pynite"] = abs(dy) * 1e3
    check(1, "Pynite tip deflection", abs(dy), delta, unit="m")
    check(1, "Pynite wall reaction", frame.nodes["N1"].RxnFY["Combo 1"], P, unit="N")
    check(1, "Pynite wall moment", frame.nodes["N1"].RxnMZ["Combo 1"], P * L, unit="N*m")

OK    Pynite tip deflection: 0.0256 vs 0.0256 m  (rel error 0.0e+00, tol 1e-09)
OK    Pynite wall reaction: 1000 vs 1000 N  (rel error 2.3e-16, tol 1e-09)
OK    Pynite wall moment: 2000 vs 2000 N*m  (rel error 1.1e-16, tol 1e-09)


In [7]:
# anaStruct reports support reactions with the opposite sign: Fy = -1000 N here.
with guarded(1, "anaStruct"):
    from anastruct import SystemElements
    ss = SystemElements(EA=E * A, EI=E * I)
    ss.add_element([[0, 0], [L, 0]])
    ss.add_support_fixed(1)
    ss.point_load(2, Fy=-P)
    ss.solve()
    tip = ss.get_node_displacements(2)
    wall = ss.get_node_results_system(1)
    TIP["anaStruct"] = abs(tip["uy"]) * 1e3
    check(1, "anaStruct tip deflection", abs(tip["uy"]), delta, unit="m")
    check(1, "anaStruct tip rotation", abs(tip["phi_z"]), theta, unit="rad")
    check(1, "anaStruct wall reaction", abs(wall["Fy"]), P, unit="N")
    check(1, "anaStruct wall moment", abs(wall["Tz"]), P * L, unit="N*m")

OK    anaStruct tip deflection: 0.0256 vs 0.0256 m  (rel error 1.4e-16, tol 1e-09)
OK    anaStruct tip rotation: 0.0192 vs 0.0192 rad  (rel error 0.0e+00, tol 1e-09)
OK    anaStruct wall reaction: 1000 vs 1000 N  (rel error 2.3e-16, tol 1e-09)
OK    anaStruct wall moment: 2000 vs 2000 N*m  (rel error 2.3e-16, tol 1e-09)


In [8]:
# SymBeam: symbolic piecewise solution. solve() prints a report unless output=False.
with guarded(1, "SymBeam"):
    from symbeam import beam as symbeam_beam
    sb = symbeam_beam(L, x0=0)
    sb.add_support(0, "fixed")
    sb.add_point_load(L, -P)
    sb.set_young(0, L, E)
    sb.set_inertia(0, L, I)
    sb.solve(output=False)
    seg = sb.segments[-1]
    xsym = list(seg.deflection.free_symbols)[0]
    v_tip = float(seg.deflection.subs(xsym, L))
    m_wall = float(sb.segments[0].bending_moment.subs(xsym, 0))
    TIP["SymBeam"] = abs(v_tip) * 1e3
    check(1, "SymBeam tip deflection", abs(v_tip), delta, unit="m")
    check(1, "SymBeam wall moment", abs(m_wall), P * L, unit="N*m")

OK    SymBeam tip deflection: 0.0256 vs 0.0256 m  (rel error 1.4e-16, tol 1e-09)
OK    SymBeam wall moment: 2000 vs 2000 N*m  (rel error 0.0e+00, tol 1e-09)


In [9]:
# OpenSeesPy: one elastic beam-column element, linear static analysis.
with guarded(1, "OpenSeesPy"):
    import openseespy.opensees as ops
    ops.wipe()
    ops.model("basic", "-ndm", 2, "-ndf", 3)
    ops.node(1, 0.0, 0.0)
    ops.node(2, L, 0.0)
    ops.fix(1, 1, 1, 1)
    ops.geomTransf("Linear", 1)
    ops.element("elasticBeamColumn", 1, 1, 2, A, E, I, 1)
    ops.timeSeries("Linear", 1)
    ops.pattern("Plain", 1, 1)
    ops.load(2, 0.0, -P, 0.0)
    ops.system("BandGeneral"); ops.numberer("RCM"); ops.constraints("Plain")
    ops.integrator("LoadControl", 1.0); ops.algorithm("Linear"); ops.analysis("Static")
    ops.analyze(1)
    ops.reactions()
    v_tip = ops.nodeDisp(2, 2)
    TIP["OpenSeesPy"] = abs(v_tip) * 1e3
    check(1, "OpenSeesPy tip deflection", abs(v_tip), delta, unit="m")
    check(1, "OpenSeesPy wall reaction", ops.nodeReaction(1, 2), P, unit="N")
    check(1, "OpenSeesPy wall moment", ops.nodeReaction(1, 3), P * L, unit="N*m")

OK    OpenSeesPy tip deflection: 0.0256 vs 0.0256 m  (rel error 1.4e-16, tol 1e-09)
OK    OpenSeesPy wall reaction: 1000 vs 1000 N  (rel error 5.7e-16, tol 1e-09)
OK    OpenSeesPy wall moment: 2000 vs 2000 N*m  (rel error 3.4e-16, tol 1e-09)


In [10]:
# CALFEM: assemble K yourself. DOFs are numbered from 1 in edof and bc; NumPy indexes from 0.
with guarded(1, "CALFEM"):
    import calfem.core as cfc
    edof = np.array([[1, 2, 3, 4, 5, 6],
                     [4, 5, 6, 7, 8, 9]])
    ex = [[0.0, L / 2], [L / 2, L]]
    K_calfem = np.zeros((9, 9))
    for dofs, x in zip(edof, ex):
        cfc.assem(dofs, K_calfem, cfc.beam2e(x, [0.0, 0.0], [E, A, I]))
    f_calfem = np.zeros((9, 1))
    f_calfem[7] = -P                       # DOF 8: v at the tip
    a, r = cfc.solveq(K_calfem, f_calfem, np.array([1, 2, 3]))
    TIP["CALFEM"] = abs(a[7, 0]) * 1e3
    check(1, "CALFEM tip deflection", abs(a[7, 0]), delta, unit="m")
    check(1, "CALFEM wall reaction", r[1, 0], P, unit="N")
    check(1, "CALFEM wall moment", r[2, 0], P * L, unit="N*m")
    check_same(1, "CALFEM K symmetric", bool(np.allclose(K_calfem, K_calfem.T)), True)

OK    CALFEM tip deflection: 0.0256 vs 0.0256 m  (rel error 4.1e-16, tol 1e-09)
OK    CALFEM wall reaction: 1000 vs 1000 N  (rel error 0.0e+00, tol 1e-09)
OK    CALFEM wall moment: 2000 vs 2000 N*m  (rel error 2.3e-16, tol 1e-09)
OK    CALFEM K symmetric: True vs True


In [11]:
# EasyFEA: the beam model takes a meshed cross-section; its Iz is computed from that mesh.
with guarded(1, "EasyFEA"):
    from EasyFEA import Mesher, ElemType, Models, Simulations
    from EasyFEA.Geoms import Point, Line, Domain
    section = Mesher().Mesh_2D(Domain(Point(-b / 2, -h / 2), Point(b / 2, h / 2), b / 10), [], ElemType.QUAD4)
    efbeam = Models.Beam.Isotropic(2, Line(Point(0, 0), Point(L, 0), L / 10), section, E, nu)
    mesh = Mesher().Mesh_Beams([efbeam], ElemType.SEG2)
    simu = Simulations.Beam(mesh, efbeam)
    wall_node = mesh.Nodes_Point(Point(0, 0))
    tip_node = mesh.Nodes_Point(Point(L, 0))
    simu.add_dirichlet(wall_node, [0, 0, 0], ["x", "y", "rz"])
    simu.add_neumann(tip_node, [-P], ["y"])
    simu.Solve()
    v_tip = simu.Result("uy")[tip_node[0]]
    TIP["EasyFEA"] = abs(v_tip) * 1e3
    check(1, "EasyFEA section Iz", efbeam.Iz, I, unit="m^4")
    check(1, "EasyFEA tip deflection", abs(v_tip), delta, unit="m")
    check(1, "EasyFEA wall reaction", simu.Result("fy")[wall_node[0]], P, unit="N")
    check(1, "EasyFEA wall moment", simu.Result("cz")[wall_node[0]], P * L, unit="N*m")

OK    EasyFEA section Iz: 5.20833e-07 vs 5.20833e-07 m^4  (rel error 0.0e+00, tol 1e-09)
OK    EasyFEA tip deflection: 0.0256 vs 0.0256 m  (rel error 1.3e-13, tol 1e-09)
OK    EasyFEA wall reaction: 1000 vs 1000 N  (rel error 2.3e-13, tol 1e-09)
OK    EasyFEA wall moment: 2000 vs 2000 N*m  (rel error 1.5e-13, tol 1e-09)


In [12]:
# scikit-fem: the weak form of the Euler-Bernoulli beam, with cubic Hermite elements.
with guarded(1, "scikit-fem"):
    from skfem import MeshLine, Basis, ElementLineHermite, BilinearForm, condense, solve
    from skfem.helpers import dd, ddot

    @BilinearForm
    def bending(u, v, w):
        return E * I * ddot(dd(u), dd(v))            # integral of EI w'' v''

    line = MeshLine(np.linspace(0.0, L, 3)).with_boundaries({"wall": lambda x: x[0] == 0.0})
    hb = Basis(line, ElementLineHermite())
    K_beam = bending.assemble(hb)
    f_beam = np.zeros(hb.N)
    tip_dof = hb.nodal_dofs[0, -1]                    # row 0: deflection, row 1: slope
    f_beam[tip_dof] = -P
    w_beam = solve(*condense(K_beam, f_beam, D=hb.get_dofs("wall")))
    reactions = K_beam @ w_beam - f_beam
    TIP["scikit-fem"] = abs(w_beam[tip_dof]) * 1e3
    check(1, "scikit-fem tip deflection", abs(w_beam[tip_dof]), delta, unit="m")
    check(1, "scikit-fem wall reaction", reactions[hb.nodal_dofs[0, 0]], P, unit="N")
    check(1, "scikit-fem wall moment", reactions[hb.nodal_dofs[1, 0]], P * L, unit="N*m")

OK    scikit-fem tip deflection: 0.0256 vs 0.0256 m  (rel error 5.4e-16, tol 1e-09)
OK    scikit-fem wall reaction: 1000 vs 1000 N  (rel error 1.8e-15, tol 1e-09)
OK    scikit-fem wall moment: 2000 vs 2000 N*m  (rel error 0.0e+00, tol 1e-09)


In [13]:
spread = (max(TIP.values()) - min(TIP.values())) / (delta * 1e3) if TIP else float("nan")
print(f"{len(TIP)} packages, closed form {delta*1e3:.6f} mm, largest difference between any two: {spread:.1e} of it")
with pd.option_context("display.float_format", "{:.9f}".format):
    display(pd.DataFrame({"tip deflection [mm]": TIP}))

9 packages, closed form 25.600000 mm, largest difference between any two: 5.6e-13 of it


,tip deflection [mm]
SymPy,25.600000000
PlaneSections,25.600000000
Pynite,25.600000000
anaStruct,25.600000000
SymBeam,25.600000000
OpenSeesPy,25.600000000
CALFEM,25.600000000
EasyFEA,25.600000000
scikit-fem,25.600000000


## 2. Section properties of the 50 × 50 mm square

`sectionproperties` meshes the section and solves for its properties. For the square, the second
moment of area is a⁴/12. The Saint-Venant torsion constant needs the warping solution, and
Timoshenko and Goodier give it as a series for a b × d rectangle (d ≥ b):

J = (b³d / 3) · [1 − (192 b / π⁵ d) Σ tanh(nπd / 2b) / n⁵], summed over odd n.

For a square, J = 0.1406 a⁴. Units in this problem are mm.

In [14]:
with guarded(2, "sectionproperties"):
    from sectionproperties.pre.library import rectangular_section
    from sectionproperties.analysis import Section
    a_sq = 50.0                                   # side [mm]
    geom = rectangular_section(d=a_sq, b=a_sq)
    geom.create_mesh(mesh_sizes=[2.0])            # max triangle area [mm^2]
    sec = Section(geometry=geom)
    sec.calculate_geometric_properties()
    sec.calculate_warping_properties()
    ixx, iyy, ixy = sec.get_ic()
    series = sum(math.tanh(n * math.pi / 2) / n**5 for n in range(1, 200, 2))
    J_series = a_sq**4 / 3 * (1 - 192 / math.pi**5 * series)
    check(2, "sectionproperties Ixx", ixx, a_sq**4 / 12, unit="mm^4")
    check(2, "sectionproperties Ixy", ixy, 0.0, abs_tol=1e-6, unit="mm^4")
    # The torsion constant comes from a warping FE solution on the mesh above
    check(2, "sectionproperties torsion constant J", sec.get_j(), J_series, rel_tol=1e-4, unit="mm^4")
    print(f"J / a^4 = {sec.get_j() / a_sq**4:.5f}   (series: {J_series / a_sq**4:.5f})")

OK    sectionproperties Ixx: 520833 vs 520833 mm^4  (rel error 2.3e-15, tol 1e-09)
OK    sectionproperties Ixy: 4.65661e-10 vs 0 mm^4  (abs error 4.7e-10, tol 1e-06)
OK    sectionproperties torsion constant J: 878608 vs 878606 mm^4  (rel error 2.2e-06, tol 1e-04)
J / a^4 = 0.14058   (series: 0.14058)


## 3. Thick-walled cylinder under internal pressure (Lamé)

The README's scikit-fem example: a 100 mm bore, 200 mm outside diameter, steel, 100 MPa internal
pressure, in plane strain. A quarter of the section is modeled on a 16 × 32 grid of quadratic
triangles. Lamé's solution, with A = p rᵢ² / (r₀² − rᵢ²):

- hoop stress σθ(r) = A (1 + r₀² / r²), 166.67 MPa at the bore
- radial displacement, plane strain: u(r) = (1 + ν)/E · A · [(1 − 2ν) r + r₀² / r]

Tolerances come from the README's mesh study on this mesh: 3.8 × 10⁻⁴ for the displacement and
1.2 × 10⁻³ for the hoop stress. The reaction across the symmetry plane must equal p·rᵢ on any
mesh, because it is equilibrium.

In [15]:
with guarded(3, "scikit-fem Lame"):
    from skfem import MeshTri, FacetBasis, ElementTriP2, ElementVector, LinearForm
    from skfem.helpers import dot
    from skfem.models.elasticity import linear_elasticity, lame_parameters

    ri, ro, p = 0.05, 0.10, 100e6     # bore and outside radius [m], internal pressure [Pa]
    m = MeshTri.init_tensor(np.linspace(ri, ro, 17), np.linspace(0, np.pi / 2, 33))
    rr, tt = m.p
    m = MeshTri(np.array([rr * np.cos(tt), rr * np.sin(tt)]), m.t).with_boundaries({
        "bore": lambda x: np.hypot(*x) < 1.001 * ri,   # facet midpoints sit just inside the arc
        "y0": lambda x: np.isclose(x[1], 0.0),         # symmetry plane: v = 0
        "x0": lambda x: np.isclose(x[0], 0.0),         # symmetry plane: u = 0
    })
    basis = Basis(m, ElementVector(ElementTriP2()))
    bore = FacetBasis(m, basis.elem, facets="bore")
    lam, mu = lame_parameters(E, nu)                    # 3D lambda, mu: plane strain in 2D
    K = linear_elasticity(lam, mu).assemble(basis)
    f = LinearForm(lambda v, w: dot(-p * w.n, v)).assemble(bore)
    D = np.concatenate([basis.get_dofs("y0").all("u^2"), basis.get_dofs("x0").all("u^1")])
    u = solve(*condense(K, f, D=D))

    Al = p * ri**2 / (ro**2 - ri**2)
    node = np.flatnonzero(np.isclose(m.p[0], ri) & np.isclose(m.p[1], 0.0))[0]
    check(3, "bore radial displacement", u[basis.nodal_dofs[0, node]],
          (1 + nu) / E * Al * ((1 - 2 * nu) * ri + ro**2 / ri), rel_tol=1e-3, unit="m")
    check(3, "reaction across y = 0 (equilibrium)", -(K @ u - f)[basis.get_dofs("y0").all("u^2")].sum(),
          p * ri, unit="N/m")

    # Hoop stress at the bore, from the displacement gradient on the bore facets
    g = bore.interpolate(u).grad
    eps = 0.5 * (g + np.transpose(g, (1, 0, 2, 3)))
    tr = eps[0, 0] + eps[1, 1]
    sxx, syy, sxy = 2 * mu * eps[0, 0] + lam * tr, 2 * mu * eps[1, 1] + lam * tr, 2 * mu * eps[0, 1]
    xq = bore.global_coordinates().value
    th = np.arctan2(xq[1], xq[0])
    s_hoop = sxx * np.sin(th)**2 + syy * np.cos(th)**2 - 2 * sxy * np.sin(th) * np.cos(th)
    check(3, "peak hoop stress at the bore", s_hoop.max(), Al * (1 + ro**2 / ri**2), rel_tol=5e-3, unit="Pa")

OK    bore radial displacement: 4.76485e-05 vs 4.76667e-05 m  (rel error 3.8e-04, tol 1e-03)
OK    reaction across y = 0 (equilibrium): 5e+06 vs 5e+06 N/m  (rel error 5.6e-14, tol 1e-09)
OK    peak hoop stress at the bore: 1.669e+08 vs 1.66667e+08 Pa  (rel error 1.4e-03, tol 5e-03)


## 4. Mass-spring-damper

m = 1 kg, c = 2 N·s/m, k = 100 N/m, released from x₀ = 10 mm at rest. That gives ωn = 10 rad/s
and ζ = 0.1. The underdamped closed form is

x(t) = x₀ e^(−ζωn t) [cos ωd t + ζ/√(1 − ζ²) sin ωd t], with ωd = ωn √(1 − ζ²).

A unit step in force overshoots by exp(−πζ/√(1 − ζ²)) = 72.925%, and the static gain is 1/k.

In [16]:
m_, c_, k_, x0 = 1.0, 2.0, 100.0, 0.01     # [kg, N*s/m, N/m, m]
wn = math.sqrt(k_ / m_)
zeta = c_ / (2 * math.sqrt(k_ * m_))
wd = wn * math.sqrt(1 - zeta**2)

def x_exact(t):
    return x0 * np.exp(-zeta * wn * t) * (np.cos(wd * t) + zeta / math.sqrt(1 - zeta**2) * np.sin(wd * t))

overshoot = math.exp(-math.pi * zeta / math.sqrt(1 - zeta**2))
print(f"wn = {wn} rad/s, zeta = {zeta}, overshoot = {overshoot:.5%}")

wn = 10.0 rad/s, zeta = 0.1, overshoot = 72.92476%


In [17]:
with guarded(4, "SciPy solve_ivp"):
    from scipy.integrate import solve_ivp
    sol = solve_ivp(lambda t, y: [y[1], -(c_ * y[1] + k_ * y[0]) / m_], (0, 3), [x0, 0.0],
                    method="DOP853", rtol=1e-10, atol=1e-14, dense_output=True)
    tt = np.linspace(0, 3, 3001)
    check(4, "solve_ivp max |x - exact| / x0", np.abs(sol.sol(tt)[0] - x_exact(tt)).max() / x0, 0.0, abs_tol=1e-8)

OK    solve_ivp max |x - exact| / x0: 9.74746e-11 vs 0   (abs error 9.7e-11, tol 1e-08)


In [18]:
# python-control. step_info uses its own coarse grid (a README gotcha), so the overshoot
# comes from step_response on a fine, evenly spaced grid.
with guarded(4, "python-control"):
    import control as ct
    plant = ct.tf(1, [m_, c_, k_])
    wn_ct, zeta_ct, _ = ct.damp(plant, doprint=False)
    check(4, "control damp: natural frequency", wn_ct[0], wn, unit="rad/s")
    check(4, "control damp: damping ratio", zeta_ct[0], zeta)
    check(4, "control dcgain", ct.dcgain(plant), 1 / k_, unit="m/N")
    T_fine = np.linspace(0, 6, 600001)
    _, y = ct.step_response(plant, T=T_fine)
    check(4, "control step overshoot", y.max() * k_ - 1, overshoot, rel_tol=1e-6)

OK    control damp: natural frequency: 10 vs 10 rad/s  (rel error 1.8e-16, tol 1e-09)
OK    control damp: damping ratio: 0.1 vs 0.1   (rel error 2.8e-16, tol 1e-09)
OK    control dcgain: 0.01 vs 0.01 m/N  (rel error 0.0e+00, tol 1e-09)


OK    control step overshoot: 0.729248 vs 0.729248   (rel error 1.9e-10, tol 1e-06)


In [19]:
# PathSim, the README snippet: the default solver is fixed-step 2nd order, so pass RKDP54.
with guarded(4, "PathSim"):
    from pathsim import Simulation, Connection
    from pathsim.blocks import Integrator, Amplifier, Adder, Scope, Constant, PID
    from pathsim.solvers import RKDP54
    v = Integrator(0.0)
    x = Integrator(x0)
    damp, spring, acc, scope = Amplifier(-c_ / m_), Amplifier(-k_ / m_), Adder(), Scope()
    sim = Simulation(
        blocks=[v, x, damp, spring, acc, scope],
        connections=[Connection(v, x, damp), Connection(x, spring, scope),
                     Connection(damp, acc[0]), Connection(spring, acc[1]), Connection(acc, v)],
        Solver=RKDP54, tolerance_lte_rel=1e-8, tolerance_lte_abs=1e-12, log=False)
    sim.run(3.0)
    t_ps, (x_ps,) = scope.read()
    check(4, "PathSim max |x - exact| / x0", np.abs(x_ps - x_exact(t_ps)).max() / x0, 0.0, abs_tol=1e-7)

OK    PathSim max |x - exact| / x0: 3.59593e-09 vs 0   (abs error 3.6e-09, tol 1e-07)


## 5. PID position loop

The same plant, driven by a PID controller to a 10 mm setpoint:
C(s) = Kp + Ki/s + Kd·s/(1 + s/f_max), with Kp = 350, Ki = 300, Kd = 50 and f_max = 500 rad/s.
PathSim simulates the block diagram in the time domain. python-control computes the closed-loop
transfer function CP/(1 + CP) and its step response. The two share no code. PathSim's `f_max` is
in rad/s, not Hz (a README gotcha).

In [20]:
Kp, Ki, Kd, f_max, x_set = 350.0, 300.0, 50.0, 500.0, 0.01

def pid_reference(t):
    """python-control's closed-loop step response, on a fine grid, interpolated to t."""
    from scipy.interpolate import CubicSpline
    s = ct.tf("s")
    loop = ct.feedback((Kp + Ki / s + Kd * s / (1 + s / f_max)) / (m_ * s**2 + c_ * s + k_), 1)
    t_fine = np.linspace(0, t.max(), 300001)    # forced_response needs evenly spaced times
    _, y_fine = ct.step_response(loop, T=t_fine)
    return x_set * CubicSpline(t_fine, y_fine)(t)

with guarded(5, "PathSim PID loop"):
    sp_, err = Constant(x_set), Adder("+-")
    pid = PID(Kp=Kp, Ki=Ki, Kd=Kd, f_max=f_max)
    v, x = Integrator(0.0), Integrator(0.0)
    damp, spring, force, acc, scope = Amplifier(-c_ / m_), Amplifier(-k_ / m_), Amplifier(1 / m_), Adder(), Scope()
    sim = Simulation(
        blocks=[sp_, err, pid, v, x, damp, spring, force, acc, scope],
        connections=[Connection(sp_, err[0]), Connection(x, err[1], spring, scope), Connection(err, pid),
                     Connection(pid, force), Connection(force, acc[2]), Connection(v, x, damp),
                     Connection(damp, acc[0]), Connection(spring, acc[1]), Connection(acc, v)],
        Solver=RKDP54, tolerance_lte_rel=1e-8, tolerance_lte_abs=1e-12, log=False)
    sim.run(3.0)
    t_pid, (x_pid,) = scope.read()
    check(5, "PathSim vs python-control, max |difference|", np.abs(x_pid - pid_reference(t_pid)).max(),
          0.0, abs_tol=1e-9, unit="m")

OK    PathSim vs python-control, max |difference|: 2.02827e-12 vs 0 m  (abs error 2.0e-12, tol 1e-09)


## 6. Desk calculations

The README's SciPy examples, each against its closed form.

- **Shaft sizing** (`brentq`, `pint`, `uncertainties`, `handcalcs`): a solid shaft under
  M = 12 000 lbf·in of bending, with Sy = 50 kpsi and n = 2. From σ = 32M/πd³ and n = Sy/σ,
  d = (32Mn / πSy)^(1/3). This problem is in inch-pound units, as in Shigley.
- **Principal stresses** (`linalg.eigh`): Shigley Ex. 3-4, σx = 80 MPa and τxy = 50 MPa, so
  σ = 40 ± √(40² + 50²) MPa.
- **Castigliano** (`integrate.quad`): the Problem 1 cantilever, δ = ∫ M (∂M/∂P) / EI dx with M = Px.
- **S–N fit** (`optimize.curve_fit`): four points generated exactly on a Basquin curve,
  S = a·N^b. The fit must recover a and b, and agree with a straight-line fit in log-log space.
- **Weibull life** (`stats.weibull_min`): B10 = scale · (−ln 0.9)^(1/shape).

In [21]:
from scipy import optimize, integrate, linalg, stats
import pint
from uncertainties import ufloat

u = pint.UnitRegistry()
M_shaft = (12_000 * u.lbf * u.inch).to("N*m").magnitude    # bending moment [N*m]
Sy = (50 * u.kpsi).to("Pa").magnitude                      # yield strength [Pa]
n_target = 2.0                                             # design factor [-]
check(6, "pint: 50 kpsi in MPa", Sy / 1e6, 50e3 * 6894.757293168361 / 1e6, unit="MPa")

with guarded(6, "brentq shaft sizing"):
    d_shaft = optimize.brentq(lambda d: Sy / (32 * M_shaft / (math.pi * d**3)) - n_target, 1e-3, 1.0)
    d_closed = (32 * M_shaft * n_target / (math.pi * Sy)) ** (1 / 3)
    check(6, "brentq shaft diameter", d_shaft, d_closed, rel_tol=1e-9, unit="m")
    print(f"d = {(d_shaft * u.m).to('inch'):.4f}")

with guarded(6, "uncertainties"):
    # d = 1.6973 in +/- 0.001 in: first-order propagation gives u(sigma)/sigma = 3 u(d)/d
    d_u = ufloat(1.6973, 0.001)                             # [in]
    sigma_u = 32 * 12_000 / (math.pi * d_u**3)              # [psi]
    check(6, "uncertainties: relative u(sigma)", sigma_u.s / sigma_u.n, 3 * 0.001 / 1.6973)

OK    pint: 50 kpsi in MPa: 344.738 vs 344.738 MPa  (rel error 3.3e-16, tol 1e-09)
OK    brentq shaft diameter: 0.0431103 vs 0.0431103 m  (rel error 1.6e-14, tol 1e-09)
d = 1.6973 inch
OK    uncertainties: relative u(sigma): 0.00176751 vs 0.00176751   (rel error 1.2e-16, tol 1e-09)


In [22]:
# handcalcs renders the next cell as a hand calculation. Inch-pound units: lbf*in, psi, in.
import handcalcs.render
from math import pi
M = 12000.0
S_y = 50000.0
d = (32 * M * 2.0 / (pi * S_y)) ** (1 / 3)

In [23]:
%%render
sigma = 32 * M / (pi * d**3)
n = S_y / sigma

<IPython.core.display.Latex object>

In [24]:
check(6, "handcalcs: rendered design factor", n, 2.0)

with guarded(6, "eigh principal stresses"):
    stress = np.array([[80.0, 50.0, 0.0], [50.0, 0.0, 0.0], [0.0, 0.0, 0.0]])   # [MPa]
    principal = linalg.eigh(stress, eigvals_only=True)[::-1]
    r_mohr = math.hypot(40.0, 50.0)
    check(6, "eigh sigma 1", principal[0], 40.0 + r_mohr, unit="MPa")
    check(6, "eigh sigma 3", principal[2], 40.0 - r_mohr, unit="MPa")
    check(6, "eigh max shear", (principal[0] - principal[2]) / 2, r_mohr, unit="MPa")

with guarded(6, "quad Castigliano"):
    delta_c, _ = integrate.quad(lambda x: (P * x) * x / (E * I), 0, L)
    check(6, "quad Castigliano tip deflection", delta_c, delta, unit="m")

with guarded(6, "curve_fit Basquin"):
    a_true, b_true = 2.262e5, -0.1274
    N_cyc = np.array([1e3, 1e4, 1e5, 1e6])
    S_amp = a_true * N_cyc**b_true
    (a_fit, b_fit), _ = optimize.curve_fit(lambda N, a, b: a * N**b, N_cyc, S_amp, p0=(1e5, -0.1))
    b_log, log_a = np.polyfit(np.log(N_cyc), np.log(S_amp), 1)
    check(6, "curve_fit Basquin a", a_fit, a_true, rel_tol=1e-6, unit="psi")
    check(6, "curve_fit Basquin b", b_fit, b_true, rel_tol=1e-6)
    check(6, "curve_fit vs log-log polyfit, b", b_fit, b_log, rel_tol=1e-6)

with guarded(6, "Weibull B10"):
    shape, scale = 2.3, 1.2e6
    check(6, "weibull_min B10 life", stats.weibull_min(shape, scale=scale).ppf(0.10),
          scale * (-math.log(0.9)) ** (1 / shape), unit="cycles")

OK    handcalcs: rendered design factor: 2 vs 2   (rel error 3.3e-16, tol 1e-09)
OK    eigh sigma 1: 104.031 vs 104.031 MPa  (rel error 0.0e+00, tol 1e-09)
OK    eigh sigma 3: -24.0312 vs -24.0312 MPa  (rel error 1.5e-16, tol 1e-09)
OK    eigh max shear: 64.0312 vs 64.0312 MPa  (rel error 0.0e+00, tol 1e-09)
OK    quad Castigliano tip deflection: 0.0256 vs 0.0256 m  (rel error 2.7e-16, tol 1e-09)
OK    curve_fit Basquin a: 226200 vs 226200 psi  (rel error 1.3e-16, tol 1e-06)
OK    curve_fit Basquin b: -0.1274 vs -0.1274   (rel error 0.0e+00, tol 1e-06)
OK    curve_fit vs log-log polyfit, b: -0.1274 vs -0.1274   (rel error 8.1e-15, tol 1e-06)
OK    weibull_min B10 life: 451085 vs 451085 cycles  (rel error 1.3e-16, tol 1e-09)


## 7. Rainflow counting and damage-equivalent load

A load history of 1000 full cycles between −1 and +1 contains exactly 1000 cycles of range 2.
The damage-equivalent load for Wöhler exponent m over n_eq equivalent cycles is

D_eq = [Σ nᵢ Rᵢ^m / n_eq]^(1/m) = 2 · (1000 / n_eq)^(1/m).

It equals the range itself when n_eq = 1000.

In [25]:
with guarded(7, "wetb"):
    from wetb.fatigue_tools.fatigue import eq_load
    history = np.append(np.tile([-1.0, 1.0], 1000), -1.0)
    for m_w, n_eq in [(4, 1000), (4, 100), (10, 1)]:
        got = eq_load(history, m=[m_w], neq=n_eq)[0][0]
        check(7, f"wetb eq_load, m = {m_w}, n_eq = {n_eq}", got, 2.0 * (1000 / n_eq) ** (1 / m_w))

OK    wetb eq_load, m = 4, n_eq = 1000: 2 vs 2   (rel error 1.1e-16, tol 1e-09)
OK    wetb eq_load, m = 4, n_eq = 100: 3.55656 vs 3.55656   (rel error 1.2e-16, tol 1e-09)
OK    wetb eq_load, m = 10, n_eq = 1: 3.99052 vs 3.99052   (rel error 1.1e-16, tol 1e-09)


## 8. AGMA spur-gear geometry

A 20-tooth, 20° full-depth AGMA gear with diametral pitch P_d = 10 teeth/in (inch units). The
standard proportions are d = N/P_d, addendum 1/P_d, dedendum 1.25/P_d, and base radius r cos φ.

In [26]:
with guarded(8, "mechanism"):
    import mechanism
    gear = mechanism.SpurGear(N=20, pd=10, pressure_angle=20, agma=True)
    check(8, "pitch diameter", gear.d, 20 / 10, unit="in")
    check(8, "base radius", gear.r_base, 1.0 * math.cos(math.radians(20)), unit="in")
    check(8, "addendum radius", gear.ra, 1.0 + 1 / 10, unit="in")
    check(8, "dedendum radius", gear.rb, 1.0 - 1.25 / 10, unit="in")

OK    pitch diameter: 2 vs 2 in  (rel error 0.0e+00, tol 1e-09)
OK    base radius: 0.939693 vs 0.939693 in  (rel error 0.0e+00, tol 1e-09)
OK    addendum radius: 1.1 vs 1.1 in  (rel error 0.0e+00, tol 1e-09)
OK    dedendum radius: 0.875 vs 0.875 in  (rel error 0.0e+00, tol 1e-09)


## 9. WR-90 waveguide

A 30 mm length of air-filled WR-90 (22.86 × 10.16 mm) is matched at both ends by waveguide ports.
It should transmit everything: |S₁₁| = 0 and |S₂₁| = 1. Its S₂₁ phase should be −βL, where
β = √(k₀² − (π/a)²) is the TE₁₀ propagation constant. RapidFEM's port 2 polarity adds a constant
180°, which is removed before comparing. This is the bundled `fd_wr90.py` example at three
frequencies. The tolerances reflect its λ/12 mesh.

In [27]:
with guarded(9, "RapidFEM"):
    import rapidfem as rf
    wg_a, wg_b, wg_L = 22.86e-3, 10.16e-3, 30.0e-3          # [m]
    freqs = np.array([8e9, 10e9, 12e9])                     # [Hz]
    geo = rf.Geometry(maxh=rf.lambda_maxh(f_max=freqs.max()))
    air = geo.box(wg_a, wg_b, wg_L, position=(-wg_a / 2, -wg_b / 2, 0), material=rf.Air())
    rf.RectWaveguidePort(air.faces.min(axis="z"))
    rf.RectWaveguidePort(air.faces.max(axis="z"))
    rf.PEC(*air.faces.unassigned)
    geo.mesh()
    S = np.asarray(rf.ProblemFD(geo).sweep(freqs).sparams)   # [frequency, out, in]
    beta = np.sqrt((2 * np.pi * freqs / 299_792_458.0)**2 - (np.pi / wg_a)**2)
    # measured phase = -beta*L + 180 deg (port 2 polarity), so this wraps to the residual
    phase_err = (np.angle(S[:, 1, 0], deg=True) + np.degrees(beta * wg_L)) % 360 - 180
    for i, fq in enumerate(freqs):
        check(9, f"|S11| at {fq/1e9:.0f} GHz", abs(S[i, 0, 0]), 0.0, abs_tol=1e-4)
        check(9, f"|S21| at {fq/1e9:.0f} GHz", abs(S[i, 1, 0]), 1.0, rel_tol=1e-4)
        check(9, f"S21 phase vs TE10 at {fq/1e9:.0f} GHz", phase_err[i], 0.0, abs_tol=0.01, unit="deg")

  Geometry normalized: L0 = 2.439318e-3 m (mean edge length)
  Material: tag=1, er=1.00, ur=1.00, tand=0.0000, cond=0.00e0, 4976 tets
RapidFEM - 4976 tets, 34484 DOFs
  Port 1: rectangular, tag=2, TE10, dims=(9371.47mm, 4165.10mm), er=1.0
  Port 2: rectangular, tag=3, TE10, dims=(9371.47mm, 4165.10mm), er=1.0
  Set up basis, materials and ports in 4.3ms


  Assembled E,B in 258.6ms (cached for sweep)


  rslab: Auto, factor nnz 7.51e6, est. peak 240 MB at 4 threads, 2 rhs (RAM 16875 MB, 15482 MB free), est. 3.87e9 flops


  f=8.0000e9 Hz [ 1/ 3]  1423.5ms  rslab LDLᵀ (assembly 375ms)


  f=1.0000e10 Hz [ 2/ 3]   357.4ms  rslab LDLᵀ (assembly 23ms)


OK    |S11| at 8 GHz: 3.11466e-06 vs 0   (abs error 3.1e-06, tol 1e-04)
OK    |S21| at 8 GHz: 0.999997 vs 1   (rel error 3.0e-06, tol 1e-04)
OK    S21 phase vs TE10 at 8 GHz: 0.00102509 vs 0 deg  (abs error 1.0e-03, tol 1e-02)
OK    |S11| at 10 GHz: 4.07047e-06 vs 0   (abs error 4.1e-06, tol 1e-04)
OK    |S21| at 10 GHz: 0.999997 vs 1   (rel error 3.1e-06, tol 1e-04)
OK    S21 phase vs TE10 at 10 GHz: 0.00232897 vs 0 deg  (abs error 2.3e-03, tol 1e-02)
OK    |S11| at 12 GHz: 5.94013e-06 vs 0   (abs error 5.9e-06, tol 1e-04)
OK    |S21| at 12 GHz: 0.999997 vs 1   (rel error 3.3e-06, tol 1e-04)
OK    S21 phase vs TE10 at 12 GHz: 0.00522085 vs 0 deg  (abs error 5.2e-03, tol 1e-02)


  f=1.2000e10 Hz [ 3/ 3]   360.0ms  rslab LDLᵀ (assembly 20ms)
  S-parameters extracted in 16.9ms (context 0.4ms)


## 10. The MATLAB language, through GNU Octave

CALFEM's stiffness matrix from Problem 1 is handed to Octave, which applies the supports and solves
with `\`, as a MATLAB script would. This needs GNU Octave on the PATH, which pip can't install
(Debian/Ubuntu: `sudo apt install octave`). Without it, the check is skipped and reported as not
covered.

In [28]:
octave_exe = shutil.which("octave-cli") or shutil.which("octave")
if octave_exe is None:
    for name in ["Octave solve of CALFEM K", "Octave issymmetric(K)"]:
        skip(10, name, "GNU Octave not found on the PATH")
else:
    with guarded(10, "oct2py"):
        from oct2py import Oct2Py
        with Oct2Py() as oc:
            oc.push("K", K_calfem)
            oc.push("f", f_calfem)
            a_oct = oc.eval("K(4:9, 4:9) \\ f(4:9)", verbose=False)   # free DOFs 4-9, MATLAB indexing
            check(10, "Octave solve of CALFEM K, tip deflection", abs(float(np.ravel(a_oct)[4])), delta, unit="m")
            check_same(10, "Octave issymmetric(K)", bool(oc.eval("issymmetric(K)", verbose=False)), True)
            print(oc.eval("version()", verbose=False))

octave: X11 DISPLAY environment variable not set
octave: disabling GUI features


OK    Octave solve of CALFEM K, tip deflection: 0.0256 vs 0.0256 m  (rel error 1.5e-15, tol 1e-09)
OK    Octave issymmetric(K): True vs True
8.4.0


## 11. OpenModelica → FMU → FMPy, and the FMU in a PathSim loop

The m-c-k system from Problem 4, built acausally from four Modelica Standard Library components
with no equation written by hand. OMPython drives OpenModelica, which:

1. simulates the model, compared with the closed form;
2. exports it as a model-exchange FMU, which FMPy runs, compared with the closed form;
3. exports a version with a force input and a position output, which `pathsim-fmi` places inside
   Problem 5's PID loop, compared with python-control.

The exports are model exchange, because OpenModelica's co-simulation FMUs integrate with explicit
Euler by default (a README gotcha). This needs OpenModelica's `omc`, which pip can't install. The
notebook uses a local `omc` on the PATH, or one already serving ZeroMQ at the address in the
`OMC_PORT` environment variable. The second case suits the `openmodelica/openmodelica` Docker
image, but that container must see this notebook's working folder at the same path; set
`OM_WORKDIR` to share one. Without `omc`, the checks are skipped and reported as not covered.

In [29]:
MCK_MO = """
model MCK "Mass on a spring and damper, released from x0 = 10 mm"
  import TC = Modelica.Mechanics.Translational.Components;
  TC.Fixed fixed;
  TC.Spring spring(c = 100) "N/m";
  TC.Damper damper(d = 2) "N.s/m";
  TC.Mass mass(m = 1, s(start = 0.01, fixed = true), v(start = 0, fixed = true)) "kg";
equation
  connect(fixed.flange, spring.flange_a);
  connect(fixed.flange, damper.flange_a);
  connect(spring.flange_b, mass.flange_a);
  connect(damper.flange_b, mass.flange_a);
end MCK;

model MCKPlant "The same plant with a force input F [N] and a position output x [m]"
  import TC = Modelica.Mechanics.Translational.Components;
  Modelica.Blocks.Interfaces.RealInput F;
  Modelica.Blocks.Interfaces.RealOutput x;
  TC.Fixed fixed;
  TC.Spring spring(c = 100);
  TC.Damper damper(d = 2);
  TC.Mass mass(m = 1, s(start = 0, fixed = true), v(start = 0, fixed = true));
  Modelica.Mechanics.Translational.Sources.Force force;
  Modelica.Mechanics.Translational.Sensors.PositionSensor sensor;
equation
  connect(fixed.flange, spring.flange_a);
  connect(fixed.flange, damper.flange_a);
  connect(spring.flange_b, mass.flange_a);
  connect(damper.flange_b, mass.flange_a);
  connect(force.flange, mass.flange_b);
  connect(F, force.f);
  connect(mass.flange_b, sensor.flange);
  connect(sensor.s, x);
end MCKPlant;
"""

OM_CHECKS = ["OpenModelica simulation", "FMPy model exchange", "pathsim-fmi FMU in PID loop"]
omc = None
if os.environ.get("OMC_PORT"):
    import OMPython
    omc = OMPython.OMCSessionPort(omc_port=os.environ["OMC_PORT"])
elif shutil.which("omc"):
    import OMPython
    omc = OMPython.OMCSessionLocal()

if omc is None:
    for name in OM_CHECKS:
        skip(11, name, "OpenModelica (omc) not found, and OMC_PORT is not set")
else:
    with guarded(11, "OpenModelica"):
        workdir = os.environ.get("OM_WORKDIR") or tempfile.mkdtemp(prefix="smoke_om_")
        with open(os.path.join(workdir, "MCK.mo"), "w") as fh:
            fh.write(MCK_MO)
        print(omc.sendExpression("getVersion()"))
        omc.sendExpression(f'cd("{workdir}")')
        assert omc.sendExpression('loadFile("MCK.mo")'), omc.sendExpression("getErrorString()")
        res = omc.sendExpression('simulate(MCK, stopTime = 3.0, numberOfIntervals = 3000, '
                                 'tolerance = 1e-10, outputFormat = "csv")')
        data = pd.read_csv(res["resultFile"])
        check(11, "OpenModelica simulation, max |x - exact| / x0",
              np.abs(data["mass.s"] - x_exact(data["time"].to_numpy())).max() / x0, 0.0, abs_tol=1e-6)

        from fmpy import simulate_fmu
        fmu_mck = omc.sendExpression('buildModelFMU(MCK, version = "2.0", fmuType = "me")')
        fm = simulate_fmu(fmu_mck, fmi_type="ModelExchange", stop_time=3.0,
                          relative_tolerance=1e-10, output=["mass.s"])
        check(11, "FMPy model exchange, max |x - exact| / x0",
              np.abs(fm["mass.s"] - x_exact(fm["time"])).max() / x0, 0.0, abs_tol=1e-6)

        from pathsim_fmi import ModelExchangeFMU
        fmu_plant = omc.sendExpression('buildModelFMU(MCKPlant, version = "2.0", fmuType = "me")')
        plant_fmu = ModelExchangeFMU(fmu_plant)               # ports named after the FMU: "F", "x"
        sp_, err = Constant(x_set), Adder("+-")
        pid = PID(Kp=Kp, Ki=Ki, Kd=Kd, f_max=f_max)
        scope = Scope()
        sim = Simulation(
            blocks=[sp_, err, pid, plant_fmu, scope],
            connections=[Connection(sp_, err[0]), Connection(plant_fmu["x"], err[1], scope),
                         Connection(err, pid), Connection(pid, plant_fmu["F"])],
            Solver=RKDP54, tolerance_lte_rel=1e-8, tolerance_lte_abs=1e-12, log=False)
        sim.run(3.0)
        t_fmu, (x_fmu,) = scope.read()
        check(11, "pathsim-fmi FMU in PID loop vs python-control", np.abs(x_fmu - pid_reference(t_fmu)).max(),
              0.0, abs_tol=1e-9, unit="m")

OpenModelica 1.27.1


OK    OpenModelica simulation, max |x - exact| / x0: 1.1194e-07 vs 0   (abs error 1.1e-07, tol 1e-06)


OK    FMPy model exchange, max |x - exact| / x0: 1.65396e-07 vs 0   (abs error 1.7e-07, tol 1e-06)


OK    pathsim-fmi FMU in PID loop vs python-control: 2.02827e-12 vs 0 m  (abs error 2.0e-12, tol 1e-09)


## Result

The table lists every check in this run. The statement below it is generated from that table
when the notebook runs. It is not written by hand, so the committed output is the evidence. The
cell raises an error if any check failed.

In [30]:
df = pd.DataFrame(RESULTS)
counts = df["status"].value_counts()
n_ok, n_fail, n_skip = (int(counts.get(s, 0)) for s in ("OK", "FAIL", "SKIP"))
versions_ok = bool((df[df["check"].str.endswith(" version")]["status"] == "OK").all())
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

statement = (
    f"Smoke test run {now} on Python {platform.python_version()} ({platform.system()} {platform.machine()}): "
    f"{len(df)} checks. {n_ok} matched their independent reference within tolerance, "
    f"{n_fail} failed, {n_skip} were skipped."
)
if versions_ok:
    statement += f" All {len(pins)} installed versions match their pins in requirements.txt."
if n_skip:
    skipped = df[df["status"] == "SKIP"]
    statement += " Not covered by this run: " + "; ".join(f"{c} ({r})" for c, r in zip(skipped["check"], skipped["note"])) + "."

pd.set_option("display.max_rows", 200)
display(df[["problem", "check", "status", "got", "want", "unit", "error", "tol", "note"]])
print(statement)

if n_fail:
    failed = df[df["status"] == "FAIL"]
    raise AssertionError(f"{n_fail} check(s) failed: " + "; ".join(failed["check"]))
print("all checks passed" if not n_skip else f"all checks that ran passed; {n_skip} skipped")

,problem,check,status,got,want,unit,error,tol,note
0,0,numpy version,OK,NaN,NaN,,NaN,NaN,2.4.6 vs 2.4.6
1,0,scipy version,OK,NaN,NaN,,NaN,NaN,1.17.1 vs 1.17.1
2,0,pandas version,OK,NaN,NaN,,NaN,NaN,3.0.6 vs 3.0.6
3,0,matplotlib version,OK,NaN,NaN,,NaN,NaN,3.11.2 vs 3.11.2
4,0,sympy version,OK,NaN,NaN,,NaN,NaN,1.14.0 vs 1.14.0
5,0,pint version,OK,NaN,NaN,,NaN,NaN,0.25.3 vs 0.25.3
6,0,forallpeople version,OK,NaN,NaN,,NaN,NaN,3.0.0 vs 3.0.0
7,0,uncertainties version,OK,NaN,NaN,,NaN,NaN,3.2.3 vs 3.2.3
8,0,handcalcs version,OK,NaN,NaN,,NaN,NaN,1.11.0 vs 1.11.0
9,0,planesections version,OK,NaN,NaN,,NaN,NaN,1.4.2 vs 1.4.2


Smoke test run 2026-10-18 16:18 UTC on Python 3.11.17 (Linux x86_64): 138 checks. 138 matched their independent reference within tolerance, 0 failed, 0 were skipped. All 31 installed versions match their pins in requirements.txt.
all checks passed
